# Week 5 metadata-only control

**Question.** How predictable is the selected-code target from age and ECG lead count without waveform data? This control does not identify whether a waveform model uses these variables as a shortcut. It quantifies an association in this hospitalized cohort.

The positive label is an exact token match between the semicolon-separated `ICD-10 code` metadata field and the 19 selected strings in `Data/DiseaseCode.csv` (all `Disease Type` values except `Other diseases(OD)`). Negative means *none of those selected codes*, not necessarily cardiovascular disease-free. No ECG diagnostic statement is used. We reconstruct the historical sorted-patient, seed-42 70/15/15 split, inspect only patient IDs for held-out patients, and fit on training patients only. Validation is the sole evaluation set. Three prespecified logistic models use age only, lead count only, and both. There is no hyperparameter search or test-set access.

In [1]:
from pathlib import Path
import csv, random, sys, json
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Data/AttributesDictionary.csv').is_file())
OUT=ROOT/'new work/Week5/experimental/metadata_control';SEED=42
assert Path(sys.executable).resolve()==(ROOT/'venv/bin/python').resolve()
print('Python',sys.executable,'scikit-learn logistic regression')
meta=ROOT/'Data/AttributesDictionary.csv'
with meta.open(newline='') as f:patient_ids=sorted({r['Patient_ID'] for r in csv.DictReader(f)})
random.Random(SEED).shuffle(patient_ids)
n_train=int(.70*len(patient_ids));n_val=int(.15*len(patient_ids))
train_ids=set(patient_ids[:n_train]);val_ids=set(patient_ids[n_train:n_train+n_val]);heldout=set(patient_ids[n_train+n_val:])
assert not (train_ids&val_ids or train_ids&heldout or val_ids&heldout)
codes=set(pd.read_csv(ROOT/'Data/DiseaseCode.csv').query('`Disease Type` != "Other diseases(OD)"')['ICD-10 Code'].astype(str).str.strip())
assert len(codes)==19
def tokens(s):return [v.strip().strip("'\"").strip() for v in s.split(';') if v.strip()]
train_rows=[];val_rows=[]
with meta.open(newline='') as f:
    for r in csv.DictReader(f):
        pid=r['Patient_ID']
        if pid in heldout:continue  # No held-out outcome or covariate read.
        row={'patient_id':pid,'age_years':float(r['Age'].removesuffix('d'))/365.25,
             'lead9':int(r['Lead']=='9'),
             'target':int(bool(set(tokens(r['ICD-10 code']))&codes))}
        (train_rows if pid in train_ids else val_rows).append(row)
train=pd.DataFrame(train_rows);val=pd.DataFrame(val_rows)
assert (len(train),len(val))==(9943,2169)
prior=pd.read_csv(ROOT/'new work/Week5/experimental/validation_predictions.csv')
assert np.array_equal(val.patient_id.to_numpy(),prior.patient_id.astype(str).to_numpy())
assert np.array_equal(val.target.to_numpy(),prior.target.to_numpy())
print('train',len(train),'validation',len(val),'val positives',int(val.target.sum()),'test patients reserved',len(heldout))
print('validation lead 9:',int(val.lead9.sum()),'prevalence',val.loc[val.lead9==1,'target'].mean())
print('validation lead 12:',int((1-val.lead9).sum()),'prevalence',val.loc[val.lead9==0,'target'].mean())

Python /home/souhit/Documents/Research/Pediatric_ECG/venv/bin/python scikit-learn logistic regression
train 9943 validation 2169 val positives 583 test patients reserved 1747
validation lead 9: 281 prevalence 0.5587188612099644
validation lead 12: 1888 prevalence 0.225635593220339


In [2]:
def metric(y,p):
    y=np.asarray(y,dtype=int);p=np.asarray(p);pred=(p>=.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(n=len(y),positives=int(y.sum()),prevalence=y.mean(),
                accuracy=accuracy_score(y,pred),sensitivity=recall_score(y,pred),
                specificity=tn/(tn+fp),precision=precision_score(y,pred,zero_division=0),
                f1=f1_score(y,pred,zero_division=0),auroc=roc_auc_score(y,p),
                average_precision=average_precision_score(y,p),
                tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))
feature_sets={'age_only':['age_years'],'lead_only':['lead9'],'age_and_lead':['age_years','lead9']}
predictions=pd.DataFrame({'validation_index':np.arange(len(val)),'patient_id':val.patient_id,'target':val.target,
                          'age_years':val.age_years,'lead_count':np.where(val.lead9==1,9,12)})
import joblib
metrics=[];coefficients=[]
for name,cols in feature_sets.items():
    clf=make_pipeline(StandardScaler(),LogisticRegression(C=1.0,max_iter=1000,solver='lbfgs',random_state=SEED))
    clf.fit(train[cols],train.target)
    joblib.dump(clf,OUT/f'{name}.joblib')
    p=clf.predict_proba(val[cols])[:,1]
    predictions[name+'_probability']=p
    metrics.append({'model':name,**metric(val.target,p)})
    coefficients.append({'model':name,'features':cols,'standardized_coefficients':clf[-1].coef_[0].tolist(),
                         'intercept':float(clf[-1].intercept_[0])})
for name in ['week5_resnet_unweighted','multiscale_resnet']:
    p=prior[name+'_probability'].to_numpy()
    predictions[name+'_probability']=p
    metrics.append({'model':name,**metric(val.target,p)})
metrics=pd.DataFrame(metrics);metrics.to_csv(OUT/'validation_metrics.csv',index=False)
predictions.to_csv(OUT/'validation_predictions.csv',index=False)
(OUT/'model_coefficients.json').write_text(json.dumps(coefficients,indent=2))
display(metrics.round(4))

,model,n,positives,prevalence,accuracy,sensitivity,specificity,precision,f1,auroc,average_precision,tn,fp,fn,tp
0,age_only,2169,583,0.2688,0.7312,0.0000,1.0000,0.0000,0.0000,0.7058,0.4507,1586,0,583,0
1,lead_only,2169,583,0.2688,0.7312,0.0000,1.0000,0.0000,0.0000,0.5956,0.3469,1586,0,583,0
2,age_and_lead,2169,583,0.2688,0.7363,0.1407,0.9552,0.5359,0.2228,0.7071,0.4570,1515,71,501,82
3,week5_resnet_unweighted,2169,583,0.2688,0.8165,0.4460,0.9527,0.7761,0.5664,0.8144,0.6923,1511,75,323,260
4,multiscale_resnet,2169,583,0.2688,0.8156,0.5129,0.9269,0.7205,0.5992,0.8192,0.6947,1470,116,284,299


In [3]:
# Descriptive subgroup metrics with n and positive prevalence.
sub=[]
for name in list(feature_sets)+['week5_resnet_unweighted','multiscale_resnet']:
    for group,mask in [('9 leads',val.lead9==1),('12 leads',val.lead9==0),
                       ('age <2y',val.age_years<2),('age 2–<6y',(val.age_years>=2)&(val.age_years<6)),
                       ('age 6–<12y',(val.age_years>=6)&(val.age_years<12)),('age >=12y',val.age_years>=12)]:
        if len(np.unique(val.target[mask]))!=2:continue
        sub.append({'model':name,'subgroup':group,**metric(val.target[mask],predictions.loc[mask,name+'_probability'])})
sub=pd.DataFrame(sub);sub.to_csv(OUT/'validation_subgroups.csv',index=False)
display(sub[['model','subgroup','n','positives','prevalence','sensitivity','specificity','auroc','average_precision']].round(4))

,model,subgroup,n,positives,prevalence,sensitivity,specificity,auroc,average_precision
0,age_only,9 leads,281,157,0.5587,0.0000,1.0000,0.4497,0.5056
1,age_only,12 leads,1888,426,0.2256,0.0000,1.0000,0.6663,0.3967
2,age_only,age <2y,272,145,0.5331,0.0000,1.0000,0.4345,0.4787
3,age_only,age 2–<6y,495,212,0.4283,0.0000,1.0000,0.6746,0.5841
4,age_only,age 6–<12y,802,136,0.1696,0.0000,1.0000,0.5546,0.2038
5,age_only,age >=12y,600,90,0.1500,0.0000,1.0000,0.5151,0.1580
6,lead_only,9 leads,281,157,0.5587,0.0000,1.0000,0.5000,0.5587
7,lead_only,12 leads,1888,426,0.2256,0.0000,1.0000,0.5000,0.2256
8,lead_only,age <2y,272,145,0.5331,0.0000,1.0000,0.5254,0.5462
9,lead_only,age 2–<6y,495,212,0.4283,0.0000,1.0000,0.5596,0.4684


In [4]:
# Patient-clustered descriptive intervals for the difference against the existing multi-scale checkpoint.
keys=np.unique(val.patient_id.to_numpy());by_patient={k:np.flatnonzero(val.patient_id.to_numpy()==k) for k in keys}
rng=np.random.default_rng(SEED);boot=[]
for rep in range(300):
    ix=np.concatenate([by_patient[k] for k in rng.choice(keys,size=len(keys),replace=True)])
    y=val.target.to_numpy()[ix]
    if len(np.unique(y))!=2:continue
    reference=predictions.multiscale_resnet_probability.to_numpy()[ix]
    for name in feature_sets:
        p=predictions[name+'_probability'].to_numpy()[ix]
        boot.append({'replicate':rep,'model':name,
                     'auroc_difference':roc_auc_score(y,reference)-roc_auc_score(y,p),
                     'ap_difference':average_precision_score(y,reference)-average_precision_score(y,p)})
boot=pd.DataFrame(boot)
summary=[]
for name,g in boot.groupby('model'):
    for m in ['auroc_difference','ap_difference']:
        summary.append({'model':name,'metric':'multiscale_minus_'+m,'mean':g[m].mean(),
                        'q025':g[m].quantile(.025),'q975':g[m].quantile(.975)})
summary=pd.DataFrame(summary);summary.to_csv(OUT/'paired_patient_bootstrap.csv',index=False)
display(summary.round(4))

,model,metric,mean,q025,q975
0,age_and_lead,multiscale_minus_auroc_difference,0.1125,0.0810,0.1506
1,age_and_lead,multiscale_minus_ap_difference,0.2359,0.1891,0.2844
2,age_only,multiscale_minus_auroc_difference,0.1137,0.0813,0.1521
3,age_only,multiscale_minus_ap_difference,0.2417,0.1912,0.2889
4,lead_only,multiscale_minus_auroc_difference,0.2238,0.1934,0.2561
5,lead_only,multiscale_minus_ap_difference,0.3475,0.3098,0.3859


## Interpretation and limits

**Observed on 2,169 validation records** (583 selected-code positives; 26.88% prevalence): age alone reached AUROC 0.7058 and AP 0.4507; lead count alone reached AUROC 0.5956 and AP 0.3469; age plus lead count reached AUROC 0.7071 and AP 0.4570. The existing multi-scale checkpoint, evaluated on the *same ordered records*, reached AUROC 0.8192 and AP 0.6947. At the fixed 0.5 threshold the age-only and lead-only logistic models predicted no positives; the combined model sensitivity was 0.1407 and specificity 0.9552. These threshold results should not be mistaken for lack of ranking information.

The validation prevalence of selected codes was higher among nine-lead records (55.87%, n=281) than twelve-lead records (22.56%, n=1,888). Age and lead availability therefore contain useful information about this metadata label in this hospitalized cohort. The waveform model's higher AP/AUROC indicates its predictions are not *reproduced* by this simple two-feature control. It does not prove that the waveform model ignores age-correlated or lead-configuration cues. Age and lead count are also correlated with each other and with care pathways; their associations are not causal. This one-split validation analysis was conducted after prior model selection, so bootstrap intervals are descriptive rather than independent confirmation.

No test patient outcomes or waveforms were read. The target remains exact matches to the 19 selected ICD-10 codes, not an adjudicated clinical diagnosis or ECG diagnostic statement.